In [3]:
# Go to /content (root workspace for Colab)
%cd /content

# Remove any previous broken clones
!rm -rf LLM-Research-Copilot

# ✅ Clone your real repo – NO angle brackets
!git clone https://github.com/maximerbc/LLM-Research-Copilot.git

# Move into the repo
%cd LLM-Research-Copilot

# Check contents
!ls


/content
Cloning into 'LLM-Research-Copilot'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 76 (delta 21), reused 57 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 34.19 MiB | 15.30 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/LLM-Research-Copilot
README.md  requirements.txt


In [4]:
!git branch -a
# Create local FineTune branch from the remote and check it out
!git checkout -b FineTune origin/FineTune

# Verify you're on FineTune
!git branch


* main
  remotes/origin/FineTune
  remotes/origin/HEAD -> origin/main
  remotes/origin/RAG
  remotes/origin/main
Branch 'FineTune' set up to track remote branch 'FineTune' from 'origin'.
Switched to a new branch 'FineTune'
* FineTune
  main


In [5]:
!pip install -q unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 19.4 MB/s eta 0:00:00


In [6]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # Mistral handles context well; 4096 is safe for Colab T4
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Essential for free Colab.

model, tokenizer = FastLanguageModel.from_pretrained(
    # CHANGED: Pointing to the specific 4-bit Mistral v0.3 model
    model_name = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.6: Fast Mistral patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.11.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [9]:
from datasets import load_dataset
ds_interview = load_dataset("K-areem/AI-Interview-Questions", split="train")

len(ds_interview)

4653

In [10]:
keywords = [
    # Core concepts
    "llm", "large language model", "language model", "lm",
    "transformer", "self-attention", "multi-head attention",
    "decoder-only", "encoder-only", "encoder-decoder",
    "autoregressive", "causal lm",

    # Architecture parts
    "kv cache", "position embeddings", "rotary embeddings", "rope",
    "feed-forward network", "ffn", "mlp block",
    "layernorm", "residual connection", "skip connection",
    "attention head", "query key value", "qkv", "softmax attention",

    # Training methods
    "pretraining", "pre-training", "next token prediction",
    "masked language modeling", "mlm", "causal modeling",
    "fine-tuning", "sft", "supervised fine-tuning",
    "lora", "qlora", "peft",
    "rlhf", "reinforcement learning from human feedback",
    "ppo", "reward model", "alignment",
    "instruction tuning", "instruct",

    # Scaling + compute
    "scaling laws", "compute optimal", "chinchilla",
    "kaplan", "hoffmann", "power law", "training tokens",
    "context length", "long-context", "sliding window attention",

    # Popular open models
    "gpt", "gpt-3", "gpt-4", "gpt-4o", "gpt-j", "gpt-neox",
    "llama", "llama 2", "llama3",
    "mistral", "mixtral", "gemma", "falcon", "phi",
    "bloom", "olmo", "qwen", "yi",

    # Model families & techniques
    "mixture of experts", "moe", "sparse mixture", "router", "experts",
    "dense transformer", "flash attention", "fused kernels",
    "grouped-query attention", "gqa", "multi-query attention", "mqa",
    "rope", "alibi", "positional encoding",

    # LLM capabilities
    "in-context learning", "zero-shot", "few-shot", "chain of thought",
    "cot", "reasoning", "hallucination", "grounding",
    "retrieval-augmented generation", "rag",

    # Evaluation + benchmarks
    "mmlu", "hellaswag", "truthfulqa", "gsm8k",
    "bigbench", "arc challenge", "math benchmark",

    # Data quality + tokenization
    "tokenizer", "bpe", "sentencepiece", "tiktoken",
    "corpus", "dataset", "data mixture", "data curation",

    # Distributed training + hardware
    "tensor parallel", "pipeline parallel", "fused attention",
    "gpu", "cuda", "accelerator", "mps", "a100",
    "zero optimization", "deepspeed", "fsdp", "sharded training",

    # RAG-specific
    "vector database", "embeddings", "semantic search",
    "chunking", "retrieval", "reranker", "colbert",

    # Research paper surnames (captures many LLM questions)
    "vaswani", "brown", "kaplan", "hoffmann",
    "touvron", "team llama", "team mistral",
    "lepikhin", "shazeer", "radford", "karpathy",
    "raffel", "devlin",
]


In [11]:
def is_llm_related(example):
  text = example["text"].lower()
  return any(k in text for k in keywords)

ds_interview_llm = ds_interview.filter(is_llm_related)
len(ds_interview_llm)

Filter:   0%|          | 0/4653 [00:00<?, ? examples/s]

877

In [12]:
from datasets import load_dataset # 1) Load your custom JSONL

ds_llm = load_dataset( "json", data_files="data/raw/llm_research_qa.jsonl", split="train", )

def to_mistral_chat(example):
  q = example["question"].strip()
  a = example["answer"].strip()
  text = f"[INST] {q} [/INST] {a}</s>"
  return {"text": text}

ds_llm_chat = ds_llm.map(to_mistral_chat)
len(ds_llm_chat)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

30

In [14]:
from datasets import concatenate_datasets # Your custom data already formatted as [INST]... style in 'text' # ds_llm_chat # HF subset is already [INST] style in 'text'

ds_combined = concatenate_datasets([ds_llm_chat, ds_interview_llm])

len(ds_combined)

907

In [15]:
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    raw_texts = examples["text"]
    texts = []

    for t in raw_texts:
        t = t.strip()

        # Ensure we have a single BOS at the start
        if not t.startswith("<s>"):
            t = "<s>" + t

        # Ensure we have exactly one EOS at the end
        if not t.endswith(EOS_TOKEN):
            t = t + EOS_TOKEN

        texts.append(t)

    return {"text": texts}

# Apply to the combined dataset
dataset = ds_combined.map(
    formatting_prompts_func,
    batched=True,
)

# Quick sanity check
print(dataset[0]["text"])


Map:   0%|          | 0/907 [00:00<?, ? examples/s]

<s>[INST] What is a large language model (LLM), and how is it typically pre-trained? [/INST] A large language model (LLM) is a transformer-based neural network trained on massive text corpora with a self-supervised objective, usually next-token prediction. During pre-training, the model learns to predict the next token given previous tokens across billions of examples, which implicitly teaches it syntax, world knowledge, and patterns of reasoning. This pre-training is domain-agnostic and does not yet specialize the model for any particular task.</s>


In [16]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Number of processors to use for processing the dataset
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2, # The batch size per GPU/TPU core
        gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
        warmup_steps = 5, # Few updates with low learning rate before actual training
        max_steps = 60, # Specifies the total number of training steps (batches) to run.
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc for observability
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/907 [00:00<?, ? examples/s]

In [18]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 907 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.378500
2,2.963900
3,3.103400
4,2.870300
5,2.969500
6,3.329700
7,2.645700
8,2.434900
9,2.385300
10,2.236600


In [19]:
import torch
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel

# 1️⃣ System prompt for your use-case
SYSTEM_PROMPT = (
    "You are an expert assistant specialized in modern large language model (LLM) "
    "research: Transformers, scaling laws, compute-optimal training, GPT-3, LLaMA, "
    "Mistral, mixtures of experts, and RAG systems. "
    "Answer clearly, precisely, and cite papers when helpful."
)

# 2️⃣ Question you want to test
question = "Explain the main difference between Kaplan et al. (2020) scaling laws and Hoffmann et al. (2022) compute-optimal training."

# 3️⃣ Attach the correct chat template for Mistral
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "mistral",   # or "mistral-instruct" depending on your Unsloth version
)

# Enable 2x faster inference in Unsloth
FastLanguageModel.for_inference(model)

# 4️⃣ Build chat messages
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": question},
]

# 5️⃣ Turn messages into input_ids using the template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,   # important for generation
    return_tensors = "pt",
)

# Move to the same device as the model
inputs = inputs.to(model.device)

# 6️⃣ Generate
with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 512,
        use_cache = True,
        temperature = 0.7,
        top_p = 0.9,
    )

# 7️⃣ Decode the response
response = tokenizer.batch_decode(
    outputs,
    skip_special_tokens = True,
)[0]

print(response)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


You are an expert assistant specialized in modern large language model (LLM) research: Transformers, scaling laws, compute-optimal training, GPT-3, LLaMA, Mistral, mixtures of experts, and RAG systems. Answer clearly, precisely, and cite papers when helpful. Explain the main difference between Kaplan et al. (2020) scaling laws and Hoffmann et al. (2022) compute-optimal training. 
The main difference between Kaplan et al. (2020) scaling laws and Hoffmann et al. (2022) compute-optimal training is that the former focuses on the relationship between model size and performance, while the latter focuses on the relationship between compute and performance.


In [2]:
# Define the save path
model_name = "llm_copilot_mistral_v03"

# Save directly to GGUF (Quantized for efficiency)
model.save_pretrained_gguf(
    model_name,
    tokenizer,
    quantization_method = "q4_k_m"  # The sweet spot for balance
)

NameError: name 'model' is not defined